# MultiSigBERT: Comparative Evaluation with Alternative Survival Models

This notebook complements the main experimental study (`multisigbert_study.ipynb`).  
Its purpose is to evaluate alternative survival modeling approaches on the exact same dataset and preprocessing pipeline, ensuring a fair and controlled comparison.

We consider a longitudinal setting where each patient is represented by a sequence of time-stamped sentence embeddings extracted from clinical reports.

| Patient | date $t\in[L-w,L]$ | Sentence embedding $v \in \mathbb{R}^q$ | Event indicator | Survival time |
|:--------|:---------:|:----------------------------------------:|:----------------:|:--------------:|
| $i$ | $\begin{aligned} &t_1 \\ &\vdots \\ &t_N \end{aligned}$ | $\begin{aligned} &v^i_{t_1} \\ &\vdots \\ &v^i_{t_N} \end{aligned}$ | $\delta_i(L) \in \{0,1\}$ | $R_i = T_i-L \ge 0$ |

Each patient $i$ is therefore described by:
- A sequence of observation times $(t_1, \dots, t_N)$,
- A corresponding sequence of sentence embeddings $(v^i_{t_k})_{k=1}^N$,
- An event indicator $\delta_i$,
- A survival time $T_i$.

The following competing models will be benchmarked under identical data splits and evaluation metrics:

1. **DeepSurv**  
   (https://arxiv.org/abs/1606.00931)  
   A neural extension of the Cox proportional hazards model trained via partial likelihood.  
   In this study, patient trajectories are summarized into fixed-length representations (e.g., mean pooling of longitudinal embeddings), resulting in a static baseline model.

2. **CoxTime**  
   (https://jmlr.org/papers/v20/18-424.html, https://github.com/havakv/pycox)  
   A neural, non-proportional hazards extension of the Cox model that learns time-dependent risk transformations.

3. **CoxTimeVaryingFitter**  
   (https://lifelines.readthedocs.io/)  
   A classical extended Cox model supporting time-dependent covariates in counting-process formulation.  
   This serves as a strong statistical baseline for longitudinal survival modeling.

All models are trained and evaluated under identical data splits and performance metrics in order to isolate architectural differences.

In [1]:
import types
import sys
from numbers import Real, Integral

# Create a fake module to emulate 'sklearn.utils._param_validation'
# (used by skglm in newer versions of scikit-learn, >=1.3)
param_validation = types.ModuleType("sklearn.utils._param_validation")

# Define a minimal replacement for Interval used in _parameter_constraints
class Interval:
    def __init__(self, dtype, left, right, closed="neither"):
        self.dtype = dtype
        self.left = left
        self.right = right
        self.closed = closed

# Define a minimal replacement for StrOptions used in _parameter_constraints
class StrOptions:
    def __init__(self, options):
        self.options = set(options)

# Add the custom classes to the fake module
param_validation.Interval = Interval
param_validation.StrOptions = StrOptions

# Inject the fake module into sys.modules before skglm is imported
# This prevents skglm from raising an ImportError if sklearn < 1.3
sys.modules["sklearn.utils._param_validation"] = param_validation

In [2]:
import pandas as pd
import torch
import numpy as np
import time
import warnings
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [3]:
import os

# Add the src directory to the Python path
notebook_dir = os.path.dirname(os.path.abspath("__file__"))
src_path = os.path.abspath(os.path.join(notebook_dir, '..', 'src/multisigbert'))
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Now import our custom modules
from _utils import *
from descriptive_stats_pkg import *
from compression_pkg import *
from survival_analysis_pkg import *
from metrics_plot_results_pkg import *
from other_models import *

/Applications/anaconda3/envs/multisigbert-env/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
start_notebook = time.time()

# I) Data Importation

## MultiModal LandMark

In [5]:
df = pd.read_csv("../data/df_study_real_L36_w12.csv")

In [6]:
df_OG = convert_date_columns(df)

In [7]:
# Structured sequential covariates
seq_struc_col = ["PO", "KAR", "TA", "PL"] #["PO", "KAR", "PS", "IMC", "TA", "PL", "TAI"]

# Base columns to keep
base_cols = [
    "ID",
    "date_creation",
    "DEATH",
    "date_death",
    "date_end",
    "date_start",
    "T_days",
    "days_since_start",
    "R",
    "DEATH_L",
    "embeddings",
]

# Final column list
keep_col = base_cols + seq_struc_col

# Restrict dataframe
df_OG = df_OG[keep_col].copy()

In [8]:
df_OG_interp, structured_columns_out = interpolate_structured_sequences(
    df_OG,
    structured_columns=seq_struc_col,
    patient_id_col='ID',
    time_col = 'date_creation')

Number of patients after interpolation: 2527
Number of structured components: 4


In [9]:
df_all, time_covariates_name, _ = expand_embedding_column(
    df=df_OG_interp,
    embedding_column="embeddings",
    parse_embedding_fn=parse_embedding,
    seq_structured_columns=seq_struc_col,
    reports_only=False
)

Embedding dimension: 768
Structured dimension: 4
Total covariate dimension: 772


# II) Other model evaluation

In [10]:
results_summary = []

## 1) DeepSurv
### 1)1) Reports Only

In [11]:
df_agg = aggregate_patient_mean(df_all, seq_struc_col=seq_struc_col)

df_zeta = pd.read_csv("../data/df_gamma_L36.csv")
df_zeta = df_zeta.rename(columns={"gamma": "zeta"})

# Inner join keeps only matching patients
df_agg = df_agg.merge(df_zeta, on="ID", how="inner")

In [12]:
df_train, test_groups = make_train_test_agg(df_agg)

In [13]:
results_mean_ds = global_mean_deepsurv(df_train,test_groups, seq_struc_col=['zeta'],verbose=False, l2_penalty = 6.8e-4)

 ---------- Ridge Penalty = 0.00068 ---------- 



In [14]:
# --- C-index Jackknife CI ---
ci_c_lower, ci_c_upper = jackknife_confidence_interval(
    results_mean_ds["c_index_tests"]
)

# --- IBS Jackknife CI ---
ci_ibs_lower, ci_ibs_upper = jackknife_confidence_interval(
    results_mean_ds["ibs_tests"]
)

print("\nSummary over independent test sets")
print(f"C-index Train: {results_mean_ds['c_index_train']:.2f}")
print(f"Mean C-index test: {results_mean_ds['mean_c_index_test']:.2f}")
print(f"SD C-index test: {results_mean_ds['std_c_index_test']:.4f}")
print(f"Mean IBS test: {results_mean_ds['mean_ibs_test']:.4f}")
print(f"SD IBS test: {results_mean_ds['std_ibs_test']:.4f}")


Summary over independent test sets
C-index Train: 0.60
Mean C-index test: 0.60
SD C-index test: 0.0237
Mean IBS test: 0.1512
SD IBS test: 0.0163


In [15]:
# Append summary row
results_summary.append({
    "Model": "Mean-DeepSurv",  # adapt if needed
    "C-index Train": results_mean_ds["c_index_train"],
    "Mean C-index Test": results_mean_ds["mean_c_index_test"],
    "Std C-index Test": results_mean_ds["std_c_index_test"],
    "C-index 95% CI Lower": ci_c_lower,
    "C-index 95% CI Upper": ci_c_upper,
    "Mean IBS Test": results_mean_ds["mean_ibs_test"],
    "Std IBS Test": results_mean_ds["std_ibs_test"],
    "IBS 95% CI Lower": ci_ibs_lower,
    "IBS 95% CI Upper": ci_ibs_upper,
})

### 1)2) Reports and Sequential Structured covariates

In [16]:
seq_struc_col_zeta = seq_struc_col+["zeta"]
results_mean_ds_struct = global_mean_deepsurv(df_train,test_groups, seq_struc_col=seq_struc_col, verbose=False,l2_penalty = 6.8e-4)

 ---------- Ridge Penalty = 0.00068 ---------- 



In [17]:
# --- C-index Jackknife CI ---
ci_c_lower, ci_c_upper = jackknife_confidence_interval(
    results_mean_ds_struct["c_index_tests"]
)

# --- IBS Jackknife CI ---
ci_ibs_lower, ci_ibs_upper = jackknife_confidence_interval(
    results_mean_ds_struct["ibs_tests"]
)

print("\nSummary over independent test sets")
print(f"C-index Train: {results_mean_ds_struct['c_index_train']:.2f}")
print(f"Mean C-index test: {results_mean_ds_struct['mean_c_index_test']:.2f}")
print(f"SD C-index test: {results_mean_ds_struct['std_c_index_test']:.4f}")
print(f"Mean IBS test: {results_mean_ds_struct['mean_ibs_test']:.4f}")
print(f"SD IBS test: {results_mean_ds_struct['std_ibs_test']:.4f}")


Summary over independent test sets
C-index Train: 0.65
Mean C-index test: 0.65
SD C-index test: 0.0429
Mean IBS test: 0.1490
SD IBS test: 0.0158


In [18]:
# Append summary row
results_summary.append({
    "Model": "Mean-DeepSurv + Structured",
    "C-index Train": results_mean_ds_struct["c_index_train"],
    "Mean C-index Test": results_mean_ds_struct["mean_c_index_test"],
    "Std C-index Test": results_mean_ds_struct["std_c_index_test"],
    "C-index 95% CI Lower": ci_c_lower,
    "C-index 95% CI Upper": ci_c_upper,
    "Mean IBS Test": results_mean_ds_struct["mean_ibs_test"],
    "Std IBS Test": results_mean_ds_struct["std_ibs_test"],
    "IBS 95% CI Lower": ci_ibs_lower,
    "IBS 95% CI Upper": ci_ibs_upper,
})

## 2) CoxTime
### 2)1) Reports Only

In [19]:
df_train, test_groups = make_train_test(df_all)

In [20]:
c_index_train, c_index_test_list, ibs_list = apply_coxtime(
    df_train=df_train,
    test_groups=test_groups,
    time_covariates_name=time_covariates_name,
    reports_only=True,
    verbose=False
)

In [21]:
# --- C-index Jackknife CI ---
ci_c_lower, ci_c_upper = jackknife_confidence_interval(
    c_index_test_list
)

# --- IBS Jackknife CI ---
ci_ibs_lower, ci_ibs_upper = jackknife_confidence_interval(
    ibs_list
)

print(f"C-index train: {c_index_train:.2f}")
print(f"Mean C-index test: {np.mean(c_index_test_list):.2f}")
print(f"SD C-index test: {np.std(c_index_test_list):.4f}")
print(f"Mean IBS test: {np.mean(ibs_list):.2f}")
print(f"SD IBS test: {np.std(ibs_list):.4f}")

C-index train: 0.77
Mean C-index test: 0.65
SD C-index test: 0.0124
Mean IBS test: 0.09
SD IBS test: 0.0150


In [22]:
# Append summary row
results_summary.append({
    "Model": "CoxTime",
    "C-index Train": c_index_train,
    "Mean C-index Test": np.mean(c_index_test_list),
    "Std C-index Test": np.std(c_index_test_list),
    "C-index 95% CI Lower": ci_c_lower,
    "C-index 95% CI Upper": ci_c_upper,
    "Mean IBS Test": np.mean(ibs_list),
    "Std IBS Test": np.std(ibs_list),
    "IBS 95% CI Lower": ci_ibs_lower,
    "IBS 95% CI Upper": ci_ibs_upper,
})

### 2)2) Reports and Sequential Structured covariates

In [23]:
c_index_train, c_index_test_list, ibs_list = apply_coxtime(
    df_train=df_train,
    test_groups=test_groups,
    time_covariates_name=time_covariates_name,
    reports_only=False,
    verbose=False
)

In [24]:
# --- C-index Jackknife CI ---
ci_c_lower, ci_c_upper = jackknife_confidence_interval(
    c_index_test_list
)

# --- IBS Jackknife CI ---
ci_ibs_lower, ci_ibs_upper = jackknife_confidence_interval(
    ibs_list
)

print(f"C-index train: {c_index_train:.2f}")
print(f"Mean C-index test: {np.mean(c_index_test_list):.2f}")
print(f"SD C-index test: {np.std(c_index_test_list):.4f}")
print(f"Mean IBS test: {np.mean(ibs_list):.2f}")
print(f"SD IBS test: {np.std(ibs_list):.4f}")

C-index train: 0.78
Mean C-index test: 0.65
SD C-index test: 0.0155
Mean IBS test: 0.09
SD IBS test: 0.0154


In [25]:
# Append summary row
results_summary.append({
    "Model": "CoxTime + Structured",
    "C-index Train": c_index_train,
    "Mean C-index Test": np.mean(c_index_test_list),
    "Std C-index Test": np.std(c_index_test_list),
    "C-index 95% CI Lower": ci_c_lower,
    "C-index 95% CI Upper": ci_c_upper,
    "Mean IBS Test": np.mean(ibs_list),
    "Std IBS Test": np.std(ibs_list),
    "IBS 95% CI Lower": ci_ibs_lower,
    "IBS 95% CI Upper": ci_ibs_upper,
})

## 3) CoxTimeVaryingFitter

### 3)1) Reports only

In [26]:
# df_train, test_groups = make_train_test(df_all)

results_ctv = global_coxtimevaryingfitter(
    df_train=df_train,
    test_groups=test_groups,
    id_col="ID",
    obs_time_col="days_since_start",
    time_col="R",
    event_col="DEATH_L",
    seq_struc_col=None,  #  or None
    penalizer=1e-3,               #  increase if convergence issues
    l1_ratio=0.0,
    verbose=False,
)

# --- C-index Jackknife CI ---
ci_c_lower, ci_c_upper = jackknife_confidence_interval(
    results_ctv["c_index_tests"]
)

# --- IBS Jackknife CI ---
ci_ibs_lower, ci_ibs_upper = jackknife_confidence_interval(
    results_ctv["ibs_tests"]
)

print("\nSummary over independent test sets")
print(f"C-index Train: {results_ctv['c_index_train']:.2f}")
print(f"Mean C-index test: {results_ctv['mean_c_index_test']:.2f}")
print(f"SD C-index test: {results_ctv['std_c_index_test']:.4f}")
print(f"Mean IBS test: {results_ctv['mean_ibs_test']:.4f}")
print(f"SD IBS test: {results_ctv['std_ibs_test']:.4f}")


Summary over independent test sets
C-index Train: 0.68
Mean C-index test: 0.62
SD C-index test: 0.0385
Mean IBS test: 0.2067
SD IBS test: 0.0339


In [27]:
# Append summary row
results_summary.append({
    "Model": "CoxTimeVaryingFitter",
    "C-index Train": results_ctv["c_index_train"],
    "Mean C-index Test": results_ctv["mean_c_index_test"],
    "Std C-index Test": results_ctv["std_c_index_test"],
    "C-index 95% CI Lower": ci_c_lower,
    "C-index 95% CI Upper": ci_c_upper,
    "Mean IBS Test": results_ctv["mean_ibs_test"],
    "Std IBS Test": results_ctv["std_ibs_test"],
    "IBS 95% CI Lower": ci_ibs_lower,
    "IBS 95% CI Upper": ci_ibs_upper,
})

### 3)2) Reports and Sequential Structured covariates

In [28]:
# seq_struc_col = ["PO", "KAR", "PS", "IMC", "TA", "PL", "TAI"] 

results_ctv = global_coxtimevaryingfitter(
    df_train=df_train,
    test_groups=test_groups,
    id_col="ID",
    obs_time_col="days_since_start",
    time_col="R",
    event_col="DEATH_L",
    seq_struc_col=seq_struc_col,  #  or None
    penalizer=1e-3,               #  increase if convergence issues
    l1_ratio=0.0,
    verbose=False,
)

lower_bound, upper_bound = jackknife_confidence_interval(
    results_ctv["c_index_tests"]
)
print(f"Jackknife CI (95%) for C-index Test: [{lower_bound:.4f}, {upper_bound:.4f}]")


lower_bound, upper_bound = jackknife_confidence_interval(
    results_ctv["ibs_tests"]
)
print(f"Jackknife CI (95%) for IBS Test: [{lower_bound:.4f}, {upper_bound:.4f}]")

print("\nSummary over independent test sets")
print(f"C-index Train: {results_ctv['c_index_train']:.2f}")
print(f"Mean C-index test: {results_ctv['mean_c_index_test']:.2f}")
print(f"SD C-index test: {results_ctv['std_c_index_test']:.4f}")
print(f"Mean IBS test: {results_ctv['mean_ibs_test']:.4f}")
print(f"SD IBS test: {results_ctv['std_ibs_test']:.4f}")

Jackknife CI (95%) for C-index Test: [0.5922, 0.6429]
Jackknife CI (95%) for IBS Test: [0.1871, 0.2310]

Summary over independent test sets
C-index Train: 0.68
Mean C-index test: 0.62
SD C-index test: 0.0388
Mean IBS test: 0.2090
SD IBS test: 0.0336


In [29]:
# Append summary row
results_summary.append({
    "Model": "CoxTimeVaryingFitter + Structured",
    "C-index Train": results_ctv["c_index_train"],
    "Mean C-index Test": results_ctv["mean_c_index_test"],
    "Std C-index Test": results_ctv["std_c_index_test"],
    "C-index 95% CI Lower": ci_c_lower,
    "C-index 95% CI Upper": ci_c_upper,
    "Mean IBS Test": results_ctv["mean_ibs_test"],
    "Std IBS Test": results_ctv["std_ibs_test"],
    "IBS 95% CI Lower": ci_ibs_lower,
    "IBS 95% CI Upper": ci_ibs_upper,
})

In [30]:
duration_notebook = time.time() - start_notebook
print(f" NoteBook total duration: {duration_notebook:.2f}s i.e. {duration_notebook / 60:.2f}min.")

 NoteBook total duration: 355.85s i.e. 5.93min.


In [31]:
df_summary = pd.DataFrame(results_summary)

In [32]:
df_summary

,Model,C-index Train,Mean C-index Test,Std C-index Test,C-index 95% CI Lower,C-index 95% CI Upper,Mean IBS Test,Std IBS Test,IBS 95% CI Lower,IBS 95% CI Upper
0,Mean-DeepSurv,0.597972,0.598575,0.023727,0.583074,0.614077,0.151241,0.016340,0.140565,0.161916
1,Mean-DeepSurv + Structured,0.645573,0.648181,0.042889,0.620160,0.676201,0.148962,0.015816,0.138629,0.159296
2,CoxTime,0.773881,0.645725,0.012369,0.637644,0.653807,0.091235,0.015009,0.081429,0.101040
3,CoxTime + Structured,0.782539,0.650597,0.015526,0.640453,0.660741,0.090603,0.015447,0.080511,0.100695
4,CoxTimeVaryingFitter,0.683122,0.619368,0.038475,0.594231,0.644505,0.206742,0.033902,0.184592,0.228892
5,CoxTimeVaryingFitter + Structured,0.680645,0.617561,0.038806,0.594231,0.644505,0.209043,0.033603,0.184592,0.228892
